In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Carrega a Gold de episodios gerada no notebook 1
df_gold = spark.table("gold_episodios")

print("Total de episodios:", df_gold.count())
print("Colunas:", df_gold.columns)
#df_gold.show(3, truncate=False)

## Feature Engineering

Todas as features comportamentais sao calculadas com dados **ANTERIORES** ao _received_time_ de cada episodio, sem data leakage

In [0]:
# Tabela de transacoes brutas para calculo de historico
df_transacoes = spark.table("silver_events") \
    .filter(F.col("event") == "transaction") \
    .select(
        "account_id",
        F.col("time_since_test_start").alias("transaction_time"),
        "amount"
    )

# Tabela de episodios anteriores para calculo de historico de ofertas
df_episodios_hist = df_gold.select(
    "account_id",
    "offer_id",
    "episodio",
    "received_time",
    "viewed_time",
    "completed_time",
    "offer_type"
)

In [0]:
# Join de cada episodio com transacoes anteriores ao seu received_time
df_feat_compra = df_gold \
    .select("account_id", "offer_id", "episodio", "received_time") \
    .join(df_transacoes, on="account_id", how="left") \
    .filter(F.col("transaction_time") < F.col("received_time")) \
    .groupBy("account_id", "offer_id", "episodio", "received_time") \
    .agg(
        F.count("amount").alias("freq_anterior"),
        F.round(F.avg("amount"), 2).alias("ticket_medio_anterior"),
        F.round(F.sum("amount"), 2).alias("gasto_total_anterior"),
        F.round(F.max("transaction_time"), 2).alias("ultima_transacao_time")
    ) \
    .withColumn(
        "recencia",
        F.round(F.col("received_time") - F.col("ultima_transacao_time"), 2)
    )


In [0]:
# Para cada episodio, conta quantas ofertas anteriores o cliente
# recebeu, visualizou e completou (apenas episodios com received_time menor)

df_hist_ofertas = df_gold \
    .select("account_id", "offer_id", "episodio", "received_time",
            "viewed_time", "completed_time") \
    .alias("atual") \
    .join(
        df_episodios_hist.alias("hist"),
        on=[
            F.col("atual.account_id") == F.col("hist.account_id"),
            F.col("hist.received_time") < F.col("atual.received_time")
        ],
        how="left"
    ) \
    .groupBy(
        F.col("atual.account_id"),
        F.col("atual.offer_id"),
        F.col("atual.episodio"),
        F.col("atual.received_time")
    ) \
    .agg(
        F.count("hist.episodio").alias("n_ofertas_anteriores"),
        F.count(F.when(F.col("hist.viewed_time").isNotNull(), 1)).alias("n_views_anteriores"),
        F.count(F.when(F.col("hist.completed_time").isNotNull(), 1)).alias("n_completadas_anteriores")
    ) \
    .withColumn(
        "taxa_view_historica",
        F.when(
            F.col("n_ofertas_anteriores") > 0,
            F.round(F.col("n_views_anteriores") / F.col("n_ofertas_anteriores"), 3)
        ).otherwise(None)
    ) \
    .withColumn(
        "taxa_conclusao_historica",
        F.when(
            F.col("n_ofertas_anteriores") > 0,
            F.round(F.col("n_completadas_anteriores") / F.col("n_ofertas_anteriores"), 3)
        ).otherwise(None)
    )

In [0]:
df_features = df_gold \
    .join(
        df_feat_compra.drop("received_time"),
        on=["account_id", "offer_id", "episodio"],
        how="left"
    ) \
    .join(
        df_hist_ofertas.drop("received_time"),
        on=["account_id", "offer_id", "episodio"],
        how="left"
    ) \
    .withColumn(
        "ticket_ratio",
        F.when(
            (F.col("min_value") > 0) & (F.col("ticket_medio_anterior").isNotNull()),
            F.round(F.col("ticket_medio_anterior") / F.col("min_value"), 3)
        ).otherwise(None)
    ) \
    .withColumn(
        "days_since_registration",
        F.when(
            F.col("registered_on").isNotNull(),
            F.datediff(
                F.date_add(F.lit("2018-01-01"), F.col("received_day").cast("int")),
                F.col("registered_on")
            )
        ).otherwise(None)
    )

print("Features geradas:", df_features.count(), "| esperado 76277")
print("Colunas:", len(df_features.columns))

# Verifica nulos nas principais features comportamentais
df_features.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ["freq_anterior", "ticket_medio_anterior", "recencia",
              "taxa_view_historica", "taxa_conclusao_historica",
              "ticket_ratio", "days_since_registration"]
]).show()

In [0]:
# ============================================================
# Tratamento de nulos e definição dos targets
# ============================================================

df_model = df_features \
    .withColumn("freq_anterior",
        F.coalesce(F.col("freq_anterior"), F.lit(0))
    ) \
    .withColumn("ticket_medio_anterior",
        F.coalesce(F.col("ticket_medio_anterior"), F.lit(0.0))
    ) \
    .withColumn("gasto_total_anterior",
        F.coalesce(F.col("gasto_total_anterior"), F.lit(0.0))
    ) \
    .withColumn("recencia",
        F.coalesce(F.col("recencia"), F.lit(999.0))
    ) \
    .withColumn("n_ofertas_anteriores",
        F.coalesce(F.col("n_ofertas_anteriores"), F.lit(0))
    ) \
    .withColumn("n_views_anteriores",
        F.coalesce(F.col("n_views_anteriores"), F.lit(0))
    ) \
    .withColumn("n_completadas_anteriores",
        F.coalesce(F.col("n_completadas_anteriores"), F.lit(0))
    ) \
    .withColumn("taxa_view_historica",
        F.coalesce(F.col("taxa_view_historica"), F.lit(0.0))
    ) \
    .withColumn("taxa_conclusao_historica",
        F.coalesce(F.col("taxa_conclusao_historica"), F.lit(0.0))
    ) \
    .withColumn("ticket_ratio",
        F.coalesce(F.col("ticket_ratio"), F.lit(0.0))
    ) \
    .withColumn("gender_encoded",
        F.when(F.col("gender") == "M", 0)
         .when(F.col("gender") == "F", 1)
         .when(F.col("gender") == "O", 2)
         .otherwise(-1)
    ) \
    .withColumn("offer_type_encoded",
        F.when(F.col("offer_type") == "bogo", 0)
         .when(F.col("offer_type") == "discount", 1)
         .when(F.col("offer_type") == "informational", 2)
         .otherwise(-1)
    )

# Target 1: successful_response
# Recebeu + visualizou dentro da validade + completou dentro da validade
# com visualizacao anterior a conclusao
df_model = df_model \
    .withColumn(
        "successful_response",
        F.when(
            (F.col("viewed_time").isNotNull()) &
            (F.col("completed_time").isNotNull()) &
            (F.col("viewed_time") <= F.col("completed_time")),
            1
        ).otherwise(0)
    ) \
    .withColumn(
        "completed_within_validity",
        F.when(F.col("completed_time").isNotNull(), 1).otherwise(0)
    )

# Episodios onde a janela de validade nao foi integralmente observada
# sao excluidos para evitar targets incompletos (censura temporal)
MAX_TIME = 29.75

df_model = df_model \
    .withColumn(
        "censurado",
        F.when(F.col("valid_until") > MAX_TIME, 1).otherwise(0)
    )

# Filtra promocionais e nao censurados
df_model_clean = df_model \
    .filter(F.col("offer_type") != "informational") \
    .filter(F.col("censurado") == 0)

# Divisao temporal por onda de disparo
df_train = df_model_clean.filter(F.col("received_day").isin([0, 7]))
df_val   = df_model_clean.filter(F.col("received_day") == 14)
df_test  = df_model_clean.filter(F.col("received_day") == 17)

print("Dataset completo (promocionais, nao censurados):", df_model_clean.count())
print("Treino (dias 0 e 7):", df_train.count())
print("Validacao (dia 14): ", df_val.count())
print("Teste (dia 17):     ", df_test.count())

print("\nDistribuicao do target successful_response:")
df_model_clean \
    .select("successful_response") \
    .groupBy("successful_response") \
    .count() \
    .orderBy("successful_response") \
    .show()

print("Distribuicao do target completed_within_validity:")
df_model_clean \
    .select("completed_within_validity") \
    .groupBy("completed_within_validity") \
    .count() \
    .orderBy("completed_within_validity") \
    .show()

print("\nEpisodios censurados (excluidos):", df_model.filter(F.col("censurado") == 1).count())

In [0]:
# ============================================================
# Preparacao das features para modelagem
# Conversao de Spark para Pandas para uso com sklearn e LightGBM
# ============================================================

# Lista de features do modelo
FEATURES = [
    # Perfil do cliente
    "age",
    "gender_encoded",
    "credit_card_limit",
    "incomplete_profile",
    "days_since_registration",
    # Comportamento de compra anterior ao envio
    "freq_anterior",
    "ticket_medio_anterior",
    "gasto_total_anterior",
    "recencia",
    # Historico de ofertas anterior ao envio
    "n_ofertas_anteriores",
    "taxa_view_historica",
    "taxa_conclusao_historica",
    # Caracteristicas da oferta
    "offer_type_encoded",
    "discount_value",
    "min_value",
    "duration",
    "n_channels",
    # Interacoes cliente x oferta
    "ticket_ratio"
]

TARGETS = ["successful_response", "completed_within_validity"]

# Seleciona apenas as colunas necessarias antes de converter para pandas
colunas_necessarias = FEATURES + TARGETS + ["received_day"]

train_pd = df_train.select(colunas_necessarias).toPandas()
val_pd   = df_val.select(colunas_necessarias).toPandas()
test_pd  = df_test.select(colunas_necessarias).toPandas()

print("Treino:", train_pd.shape)
print("Validacao:", val_pd.shape)
print("Teste:", test_pd.shape)

print("\nNulos no treino:")
print(train_pd[FEATURES].isnull().sum()[train_pd[FEATURES].isnull().sum() > 0])

print("\nDistribuicao dos targets no treino:")
print("successful_response:", train_pd["successful_response"].value_counts().to_dict())
print("completed_within_validity:", train_pd["completed_within_validity"].value_counts().to_dict())

In [0]:
# Modelo simples para estabelecer um piso de performance
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, roc_auc_score,
    average_precision_score, brier_score_loss
)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore")

TARGET = "successful_response"

X_train = train_pd[FEATURES].copy()
y_train = train_pd[TARGET].copy()

X_val = val_pd[FEATURES].copy()
y_val = val_pd[TARGET].copy()

X_test = test_pd[FEATURES].copy()
y_test = test_pd[TARGET].copy()

# Regressao Logistica exige escala padronizada e nao lida com nulos nativamente
# Imputamos a mediana do treino para age e credit_card_limit
for col in ["age", "credit_card_limit"]:
    mediana = X_train[col].median()
    X_train[col] = X_train[col].fillna(mediana)
    X_val[col]   = X_val[col].fillna(mediana)
    X_test[col]  = X_test[col].fillna(mediana)

# Pipeline com padronizacao e regressao logistica
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

pipe_lr.fit(X_train, y_train)

# Avaliacao na validacao e no teste
for nome, X, y in [("Validacao", X_val, y_val), ("Teste", X_test, y_test)]:
    probs = pipe_lr.predict_proba(X)[:, 1]
    preds = pipe_lr.predict(X)
    print(f"\n=== Regressao Logistica — {nome} ===")
    print(f"ROC-AUC:  {roc_auc_score(y, probs):.4f}")
    print(f"PR-AUC:   {average_precision_score(y, probs):.4f}")
    print(f"Brier:    {brier_score_loss(y, probs):.4f}")
    print(classification_report(y, preds, digits=3))

In [0]:
#%pip install lightgbm
#%pip install shap

In [0]:
# Modelo principal — LightGBM
# Treinado no conjunto de treino, hiperparametros ajustados
# na validacao, avaliado uma unica vez no teste

import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.metrics import classification_report

X_train_lgb = train_pd[FEATURES].copy()
y_train_lgb = train_pd[TARGET].copy()

X_val_lgb = val_pd[FEATURES].copy()
y_val_lgb = val_pd[TARGET].copy()

X_test_lgb = test_pd[FEATURES].copy()
y_test_lgb = test_pd[TARGET].copy()

# LightGBM lida nativamente com nulos, nao precisa de imputacao
model_lgb = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    verbose=-1
)

model_lgb.fit(
    X_train_lgb, y_train_lgb,
    eval_set=[(X_val_lgb, y_val_lgb)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)]
)

print("Melhor iteracao:", model_lgb.best_iteration_)

# Avaliacao na validacao e no teste
for nome, X, y in [("Validacao", X_val_lgb, y_val_lgb), ("Teste", X_test_lgb, y_test_lgb)]:
    probs = model_lgb.predict_proba(X)[:, 1]
    preds = model_lgb.predict(X)
    print(f"\n=== LightGBM — {nome} ===")
    print(f"ROC-AUC:  {roc_auc_score(y, probs):.4f}")
    print(f"PR-AUC:   {average_precision_score(y, probs):.4f}")
    print(f"Brier:    {brier_score_loss(y, probs):.4f}")
    print(classification_report(y, preds, digits=3))

In [0]:
# ============================================================
# Calibracao das probabilidades — Platt Scaling
# LightGBM tende a produzir probabilidades extremas (proximas de 0 ou 1)
# A calibracao ajusta as probabilidades para refletir melhor
# a frequencia real de eventos, essencial para o score financeiro
# ============================================================

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

# Calibracao com Platt Scaling usando o conjunto de validacao
# O modelo base ja esta treinado — calibramos so a camada de saida
calibrator = CalibratedClassifierCV(
    model_lgb,
    method="sigmoid",  # Platt Scaling
    cv="prefit"        # usa o modelo ja treinado, nao retreina
)

calibrator.fit(X_val_lgb, y_val_lgb)

# Avaliacao do modelo calibrado no teste
probs_cal = calibrator.predict_proba(X_test_lgb)[:, 1]
probs_raw = model_lgb.predict_proba(X_test_lgb)[:, 1]

print("=== Calibracao — Teste ===")
print(f"Brier Score (sem calibracao): {brier_score_loss(y_test_lgb, probs_raw):.4f}")
print(f"Brier Score (calibrado):      {brier_score_loss(y_test_lgb, probs_cal):.4f}")
print(f"PR-AUC (sem calibracao):      {average_precision_score(y_test_lgb, probs_raw):.4f}")
print(f"PR-AUC (calibrado):           {average_precision_score(y_test_lgb, probs_cal):.4f}")

# Curva de calibracao visual
fig, ax = plt.subplots(figsize=(8, 6))

for probs, label, cor in [
    (probs_raw, "LightGBM sem calibracao", "#E63946"),
    (probs_cal, "LightGBM calibrado (Platt)", "#2A6CF0")
]:
    frac_pos, mean_pred = calibration_curve(y_test_lgb, probs, n_bins=10)
    ax.plot(mean_pred, frac_pos, marker="o", label=label, color=cor)

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Calibracao perfeita")
ax.set_title("Curva de Calibracao — Teste", fontweight="bold")
ax.set_xlabel("Probabilidade prevista")
ax.set_ylabel("Frequencia real de positivos")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

%md
### Calibração das Probabilidades

A curva de calibração mostrou que o LightGBM já produz probabilidades
bem ajustadas para esse dataset — a linha vermelha segue a diagonal de
calibração perfeita com desvio pequeno. A aplicação de Platt Scaling
piorou levemente o Brier Score (0.1773 → 0.1787) sem ganho em PR-AUC.

Decisão: o modelo sem calibração adicional será usado para o score
financeiro. Em produção, recomenda-se reavaliar a calibração com dados
de um período mais longo.

In [0]:
# ============================================================
# Interpretabilidade — SHAP Values
# ============================================================

import shap

# Calcula SHAP values no conjunto de teste
explainer = shap.TreeExplainer(model_lgb)
shap_values = explainer.shap_values(X_test_lgb)

# Para classificacao binaria o LightGBM retorna lista com 2 arrays
# Pegamos o array referente a classe positiva (indice 1)
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

# Summary plot — importancia global das features
plt.figure(figsize=(10, 7))
shap.summary_plot(
    sv, X_test_lgb,
    feature_names=FEATURES,
    plot_type="bar",
    show=False
)
plt.title("Importância das Features — SHAP (LightGBM)", fontweight="bold")
plt.tight_layout()
plt.savefig("/Workspace/Users/matheusmartinez2018@gmail.com/coupon-optimization-ml/data/processed/shap_importance.png",
            dpi=150, bbox_inches="tight")
plt.show()

# Beeswarm plot — distribuicao dos valores SHAP por feature
plt.figure(figsize=(10, 7))
shap.summary_plot(
    sv, X_test_lgb,
    feature_names=FEATURES,
    show=False
)
plt.title("Distribuição dos SHAP Values — Teste", fontweight="bold")
plt.tight_layout()
plt.savefig("/Workspace/Users/matheusmartinez2018@gmail.com/coupon-optimization-ml/data/processed/shap_beeswarm.png",
            dpi=150, bbox_inches="tight")
plt.show()

### Interpretabilidade — SHAP Values

Os SHAP values revelam quais features mais influenciam a probabilidade
de resposta à oferta, e em qual direção.

**Antiguidade na plataforma** (`days_since_registration`) é o sinal mais
forte do modelo. Clientes com mais tempo de conta convertem
significativamente mais — provavelmente porque são mais engajados com
o app e já têm hábito de uso de ofertas.

**Cobertura de canais** (`n_channels`) é a segunda feature mais importante.
Ofertas veiculadas em mais canais (web, email, mobile, social)
alcançam o cliente em mais pontos de contato, aumentando a probabilidade
de resposta. Isso reforça a importância de uma estratégia omnichannel.

**Perfil financeiro** (`credit_card_limit`, `ticket_medio_anterior`, `age`)
concentra as posições 3 a 5. Clientes com maior poder aquisitivo e
ticket histórico mais alto respondem melhor às ofertas — possivelmente
porque têm menor fricção para atingir o valor mínimo de ativação.

**Um achado contraintuitivo confirmado pelo modelo:** `discount_value` tem
impacto negativo — ofertas com desconto maior convertem menos. Isso
é consistente com o que a análise descritiva já havia mostrado e sugere
que o valor do desconto sozinho não é o fator determinante de resposta.
O que importa mais é a acessibilidade da oferta (valor mínimo baixo,
prazo adequado, canais amplos) do que o tamanho do incentivo.

**Cadastros incompletos** (`incomplete_profile`) têm impacto praticamente
nulo no modelo — a ausência de dados demográficos não prediz conversão
além do que as features comportamentais já capturam.

In [0]:
# ============================================================
# Sistema de Recomendação — versão corrigida
# O score financeiro precisa equilibrar probabilidade de resposta
# e valor liquido da oferta. Normalizamos as probabilidades por
# cliente para que a escolha reflita qual oferta aquele cliente
# especifico tem MAIOR CHANCE RELATIVA de responder, ponderada
# pelo valor liquido de cada oferta
# ============================================================

linhas = []
for _, oferta in ofertas_promocionais.iterrows():
    temp = clientes_teste.copy()
    temp["offer_type_encoded"] = 0 if oferta["offer_type"] == "bogo" else 1
    temp["discount_value"]     = oferta["discount_value"]
    temp["min_value"]          = oferta["min_value"]
    temp["duration"]           = oferta["duration"]
    temp["n_channels"]         = oferta["n_channels"]
    temp["ticket_ratio"]       = np.where(
        (temp["ticket_medio_anterior"] > 0) & (oferta["min_value"] > 0),
        temp["ticket_medio_anterior"] / oferta["min_value"],
        0
    )
    temp["offer_id"]       = oferta["offer_id"]
    temp["offer_discount"] = oferta["discount_value"]
    temp["offer_min_value"] = oferta["min_value"]
    linhas.append(temp)

df_rec = pd.concat(linhas, ignore_index=True)

# Probabilidade de resposta para cada par cliente-oferta
df_rec["prob_resposta"] = model_lgb.predict_proba(df_rec[FEATURES])[:, 1]

# Normaliza a probabilidade por cliente: para cada cliente, a oferta
# com maior prob recebe score relativo mais alto independente do desconto
df_rec["prob_min"] = df_rec.groupby("account_id")["prob_resposta"].transform("min")
df_rec["prob_max"] = df_rec.groupby("account_id")["prob_resposta"].transform("max")
df_rec["prob_range"] = df_rec["prob_max"] - df_rec["prob_min"]

df_rec["prob_normalizada"] = np.where(
    df_rec["prob_range"] > 0,
    (df_rec["prob_resposta"] - df_rec["prob_min"]) / df_rec["prob_range"],
    0.5
)

# Score financeiro com probabilidade normalizada:
# P_norm(resposta) * (max(ticket_medio, min_value) - discount_value)
# A normalizacao garante que a probabilidade relativa entre ofertas
# pesa mais do que o valor absoluto do desconto
df_rec["valor_base"] = np.maximum(
    df_rec["ticket_medio_anterior"].clip(lower=0),
    df_rec["offer_min_value"]
)
df_rec["score_financeiro"] = (
    df_rec["prob_normalizada"] * (df_rec["valor_base"] - df_rec["offer_discount"])
)

# Threshold: nao recomenda se todas as probabilidades absolutas sao baixas
# Cliente com prob maxima abaixo de 0.3 recebe "nao enviar"
THRESHOLD_PROB = 0.30

df_rec["nao_enviar"] = df_rec.groupby("account_id")["prob_resposta"].transform("max") < THRESHOLD_PROB

# Melhor oferta por cliente
idx_melhor = df_rec.groupby("account_id")["score_financeiro"].idxmax()
df_recomendacao = df_rec.loc[idx_melhor].copy()
df_recomendacao["recomendacao_final"] = np.where(
    df_recomendacao["nao_enviar"],
    "nao_enviar",
    df_recomendacao["offer_id"]
)
df_recomendacao["offer_type"] = df_recomendacao["offer_type_encoded"].map({0: "bogo", 1: "discount"})

print("Distribuicao das ofertas recomendadas:")
print(df_recomendacao["recomendacao_final"].value_counts())

print("\nDistribuicao por tipo:")
print(df_recomendacao[df_recomendacao["recomendacao_final"] != "nao_enviar"]["offer_type"].value_counts())

print("\nClientes que nao receberiam oferta:", df_recomendacao["nao_enviar"].sum())

print("\nEstatisticas da probabilidade de resposta (melhor oferta por cliente):")
print(df_recomendacao["prob_resposta"].describe().round(3))

In [0]:
# Analise do ranking completo de ofertas por cliente
# Mostra que o modelo discrimina entre ofertas mesmo quando
# a recomendacao final concentra numa so

# Probabilidade media por oferta (media sobre todos os clientes do teste)
print("Probabilidade media de resposta por oferta:")
prob_media = df_rec.groupby("offer_id")["prob_resposta"].mean().sort_values(ascending=False)
print(prob_media.round(3))

# Quantas vezes cada oferta aparece como 1a, 2a, 3a opcao
print("\nRanking das ofertas por posicao:")
df_rec["rank_oferta"] = df_rec.groupby("account_id")["prob_resposta"].rank(
    ascending=False, method="min"
).astype(int)

ranking = df_rec.groupby(["rank_oferta", "offer_id"]).size().unstack(fill_value=0)
print(ranking.head(3))

# Dispersao das probabilidades por cliente
print("\nPara um cliente aleatorio, qual e a diferenca entre")
print("a melhor e a pior oferta em termos de probabilidade?")
spread = df_rec.groupby("account_id")["prob_resposta"].agg(["max", "min"])
spread["diferenca"] = spread["max"] - spread["min"]
print(spread["diferenca"].describe().round(3))

# Visualizacao: distribuicao das probabilidades por oferta
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle("Distribuição de Probabilidade de Resposta por Oferta", 
             fontsize=14, fontweight="bold")

for ax, (_, oferta) in zip(axes.flat, ofertas_promocionais.iterrows()):
    subset = df_rec[df_rec["offer_id"] == oferta["offer_id"]]["prob_resposta"]
    ax.hist(subset, bins=30, color="#2A6CF0", edgecolor="white", alpha=0.85)
    ax.axvline(subset.mean(), color="#E63946", linestyle="--", linewidth=1.5)
    ax.set_title(
        f"{oferta['offer_type'].upper()}\ndesc=R${oferta['discount_value']} "
        f"min=R${oferta['min_value']} {int(oferta['duration'])}d",
        fontsize=8, fontweight="bold"
    )
    ax.set_xlabel("P(resposta)", fontsize=7)
    ax.set_ylabel("Clientes", fontsize=7)
    ax.text(0.05, 0.88, f"média={subset.mean():.2f}",
            transform=ax.transAxes, fontsize=7, color="#E63946")

plt.tight_layout()
plt.savefig("/Workspace/Users/matheusmartinez2018@gmail.com/coupon-optimization-ml/data/processed/dist_probabilidades.png",
            dpi=150, bbox_inches="tight")
plt.show()

In [0]:
# ============================================================
# Business Case — Simulação Retrospectiva
# Compara a política histórica observada com a política do modelo
# Apresentado como simulacao, nao como ganho causal comprovado
# ============================================================

# Política histórica: todas as ofertas foram enviadas para todos os clientes
# no conjunto de teste (observado nos dados)
total_episodios_teste = len(test_pd)
total_positivos_teste = test_pd["successful_response"].sum()
taxa_conversao_historica = total_positivos_teste / total_episodios_teste

# Custo total de desconto na politica historica
# Cada episodio que foi completado gerou um custo de discount_value
df_test_spark = df_model_clean.filter(F.col("received_day") == 17)

custo_historico = df_test_spark \
    .filter(F.col("successful_response") == 1) \
    .agg(F.sum("discount_value").alias("custo")).collect()[0]["custo"]

conclusoes_sem_view = df_test_spark \
    .filter(
        (F.col("completed_within_validity") == 1) &
        (F.col("viewed_time").isNull())
    ).count()

print("=== Política Histórica (Teste — Onda dia 17) ===")
print(f"Total de episodios:              {total_episodios_teste:,}")
print(f"Conversoes (successful_response):{total_positivos_teste:,}")
print(f"Taxa de conversao:               {taxa_conversao_historica:.1%}")
print(f"Custo total de desconto:         R$ {custo_historico:,.0f}")
print(f"Conclusoes sem visualizacao:     {conclusoes_sem_view:,}")

# Política do modelo: envia oferta recomendada apenas para clientes
# com prob_resposta acima do threshold
clientes_receberiam = df_recomendacao[~df_recomendacao["nao_enviar"]]
n_envios_modelo = len(clientes_receberiam)
reducao_envios = total_episodios_teste - n_envios_modelo

# Estimativa de conversoes mantidas
# Assumimos que clientes com prob_resposta alta pelo modelo
# teriam respondido na mesma proporcao do historico
conversoes_estimadas = (clientes_receberiam["prob_resposta"] > 0.5).sum()
custo_desconto_modelo = clientes_receberiam["offer_discount"].sum()
economia_desconto = custo_historico - custo_desconto_modelo

print("\n=== Política do Modelo (Simulação Retrospectiva) ===")
print(f"Clientes que receberiam oferta:  {n_envios_modelo:,}")
print(f"Reducao de envios:               {reducao_envios:,} ({reducao_envios/total_episodios_teste:.1%})")
print(f"Custo estimado de desconto:      R$ {custo_desconto_modelo:,.0f}")
print(f"Economia estimada de desconto:   R$ {economia_desconto:,.0f}")
print(f"Reducao de custo:                {economia_desconto/custo_historico:.1%}")

print("\n=== Projecao Anual (assumindo 12 ondas de disparo) ===")
print(f"Economia anual estimada:         R$ {economia_desconto * 12:,.0f}")

print("\nNota: resultados apresentados como simulacao retrospectiva.")
print("Ganhos reais devem ser validados via teste A/B em producao.")

# Tabela de threshold para apresentacao executiva
print("\n=== Tabela de Simulacao por Threshold ===")
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
rows = []
for t in thresholds:
    envios = df_recomendacao[df_recomendacao["prob_resposta"] >= t]
    n_env = len(envios)
    custo = envios["offer_discount"].sum()
    rows.append({
        "Threshold": t,
        "Envios": n_env,
        "Reducao_envios_%": f"{(total_episodios_teste - n_env)/total_episodios_teste:.1%}",
        "Custo_desconto": f"R$ {custo:,.0f}",
        "Economia_vs_historico": f"R$ {custo_historico - custo:,.0f}"
    })

import pandas as pd
df_threshold = pd.DataFrame(rows)
print(df_threshold.to_string(index=False))

%md
## Business Case — Simulação Retrospectiva

A análise foi conduzida sobre a onda de teste (dia 17), com 10.210
episódios de oferta e taxa histórica de conversão de 38,6%.

**Conclusões sem visualização registrada**

1.135 episódios foram concluídos sem que o cliente tivesse visualizado
a oferta dentro da janela de validade. Esses casos representam possível
subsídio desnecessário — o desconto foi pago sem evidência de que a
oferta influenciou a decisão de compra. Não é possível afirmar com
certeza que essas compras teriam ocorrido sem o incentivo, mas são
candidatos prioritários para redução de custo em produção.

**Simulação por threshold de probabilidade**

O modelo permite controlar o equilíbrio entre cobertura e custo via
threshold de probabilidade mínima para envio:

| Threshold | Envios | Redução | Economia por onda | Economia anual est. |
|---|---|---|---|---|
| 0.3 | 9.368 | 8% | R$ 302 | R$ 3.624 |
| 0.4 | 8.493 | 17% | R$ 2.060 | R$ 24.720 |
| 0.5 | 7.456 | 27% | R$ 4.155 | R$ 49.860 |
| 0.6 | 6.235 | 39% | R$ 6.680 | R$ 80.160 |
| 0.7 | 4.796 | 53% | R$ 9.733 | R$ 116.796 |

A escolha do threshold depende do apetite de risco do negócio — um
threshold mais alto economiza mais desconto mas reduz a cobertura de
clientes potencialmente conversíveis. A validação do ponto ótimo deve
ser feita via teste A/B em produção, comparando grupos com thresholds
diferentes sobre uma base de clientes real.

**Limitações da simulação**

Os resultados acima são uma simulação retrospectiva sobre dados
históricos, não um experimento controlado. Não é possível afirmar com
certeza que clientes que não receberiam oferta pelo modelo teriam de
fato convertido — nem o contrário. O valor real da estratégia só pode
ser mensurado com precisão via teste A/B prospectivo.

In [0]:
# ============================================================
# Lift@K — avalia a qualidade do ranking do modelo
# Mede quantas vezes o modelo e melhor do que enviar aleatoriamente
# Util para justificar o valor do ranking, nao so da classificacao
# ============================================================

from sklearn.metrics import roc_curve

# Probabilidades e targets reais do conjunto de teste
probs_teste = model_lgb.predict_proba(X_test_lgb)[:, 1]
y_teste = y_test_lgb.values

# Ordena por probabilidade decrescente
idx_ord = np.argsort(probs_teste)[::-1]
y_ord = y_teste[idx_ord]

# Calcula lift para cada percentil
n = len(y_teste)
taxa_positivos_global = y_teste.mean()

ks_values = []
lift_values = []
percentis = np.arange(0.01, 1.01, 0.01)

conversoes_acumuladas = np.cumsum(y_ord)
conversoes_aleatorio = taxa_positivos_global * np.arange(1, n + 1)

for p in percentis:
    k = int(p * n)
    if k == 0:
        continue
    lift = (conversoes_acumuladas[k-1] / k) / taxa_positivos_global
    lift_values.append(lift)

# KS statistic
fpr, tpr, _ = roc_curve(y_teste, probs_teste)
ks_stat = np.max(tpr - fpr)

print(f"KS Statistic: {ks_stat:.4f}")
print(f"Lift@10%: {lift_values[9]:.2f}x")
print(f"Lift@20%: {lift_values[19]:.2f}x")
print(f"Lift@30%: {lift_values[29]:.2f}x")
print(f"Lift@50%: {lift_values[49]:.2f}x")

# ── Graficos finais de avaliacao ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Avaliação do Modelo — LightGBM", fontsize=14, fontweight="bold")

# 1. Curva de Lift
ax1 = axes[0]
ax1.plot(percentis[:len(lift_values)], lift_values,
         color="#2A6CF0", linewidth=2, label="LightGBM")
ax1.axhline(1, color="gray", linestyle="--", linewidth=1, label="Baseline aleatório")
ax1.fill_between(percentis[:len(lift_values)], 1, lift_values,
                 alpha=0.15, color="#2A6CF0")
ax1.set_title("Curva de Lift", fontweight="bold")
ax1.set_xlabel("Percentil da população contatada")
ax1.set_ylabel("Lift (vezes melhor que aleatório)")
ax1.legend()
ax1.grid(alpha=0.3)

# 2. Curva de Ganho Acumulado
ax2 = axes[1]
ganho_modelo = conversoes_acumuladas / conversoes_acumuladas[-1]
ganho_aleatorio = np.linspace(0, 1, n)
ax2.plot(np.linspace(0, 1, n), ganho_modelo,
         color="#2A6CF0", linewidth=2, label="LightGBM")
ax2.plot([0, 1], [0, 1], color="gray", linestyle="--",
         linewidth=1, label="Baseline aleatório")
ax2.plot([0, taxa_positivos_global, 1], [0, 1, 1],
         color="#E63946", linestyle="--", linewidth=1, label="Modelo perfeito")
ax2.set_title("Curva de Ganho Acumulado", fontweight="bold")
ax2.set_xlabel("Fração da população contatada")
ax2.set_ylabel("Fração das conversões capturadas")
ax2.legend()
ax2.grid(alpha=0.3)

# 3. Distribuicao de probabilidades por classe
ax3 = axes[2]
ax3.hist(probs_teste[y_teste == 0], bins=40, alpha=0.6,
         color="#E63946", label="Não converteu", density=True)
ax3.hist(probs_teste[y_teste == 1], bins=40, alpha=0.6,
         color="#2A6CF0", label="Converteu", density=True)
ax3.axvline(0.5, color="gray", linestyle="--", linewidth=1)
ax3.set_title("Distribuição de Probabilidades\npor Classe Real", fontweight="bold")
ax3.set_xlabel("P(resposta)")
ax3.set_ylabel("Densidade")
ax3.legend()
ax3.grid(alpha=0.3)

plt.tight_layout()
#plt.savefig("/Workspace/Users/matheusmartinez2018@gmail.com/coupon-optimization-ml/data/processed/avaliacao_modelo.#png",
#            dpi=150, bbox_inches="tight")
plt.show()
#print("Figura salva.")

%md
## Avaliação do Modelo — LightGBM

| Métrica | Regressão Logística | LightGBM | 
|---|---|---|
| ROC-AUC | 0.752 | 0.799 |
| PR-AUC | 0.633 | 0.685 |
| Brier Score | 0.205 | 0.177 |
| F1 (positivo) | 0.434 | 0.644 |
| KS Statistic | - | 0.456 |

O LightGBM supera a Regressão Logística em todas as métricas.
O ganho mais expressivo é no F1 da classe positiva (+0.21), o que
significa que o modelo identifica quem vai converter com muito mais
qualidade — essencial para um sistema de recomendação onde falsos
negativos têm custo direto (conversões perdidas).

**Curva de Lift:** contatando os 10% de clientes com maior
probabilidade prevista, o modelo captura 2x mais conversões do que
um envio aleatório. Aos 30%, ainda mantém lift de 1.77x.

**Curva de Ganho Acumulado:** contatando 40% dos clientes rankeados
pelo modelo, captura-se aproximadamente 70% de todas as conversões
do período. Isso fundamenta a estratégia de segmentação: concentrar
esforço e custo de desconto nos clientes de maior propensão.